# PDF Chatbot using LangChain and Google Gemini API

This notebook demonstrates how to build a PDF Chatbot that extracts text, creates vector embeddings, and uses an LLM to answer user queries.

In [17]:
!pip install langchain langchain-google-genai langchain-community langchain-text-splitters chromadb pypdf python-dotenv google-generativeai


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [18]:
import os
from PyPDF2 import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
import google.generativeai as genai
from langchain_community.vectorstores import Chroma
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [ ]:
# Set your Google API Key here
GOOGLE_API_KEY = ""
os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
genai.configure(api_key=GOOGLE_API_KEY)

In [20]:
def get_pdf_text(pdf_path):
    text = ""
    pdf_reader = PdfReader(pdf_path)
    for page in pdf_reader.pages:
        if page.extract_text():
            text += page.extract_text()
    return text

def get_text_chunks(text):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=10000, chunk_overlap=1000)
    chunks = text_splitter.split_text(text)
    return chunks

def get_vector_store(text_chunks):
    embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")
    vector_store = Chroma.from_texts(text_chunks, embedding=embeddings)
    return vector_store

def get_conversational_chain():
    prompt_template = """
    Answer the question as detailed as possible from the provided context, make sure to provide all the details. If the answer is not in
    provided context just say, "answer is not available in the context", don't provide the wrong answer.

    Context:
    {context}
    
    Question: 
    {question}

    Answer:
    """
    model = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.3)
    prompt = PromptTemplate(template=prompt_template, input_variables=["context", "question"])
    chain = prompt | model | StrOutputParser()
    return chain

In [21]:
# Example Usage
# 1. Place a sample PDF named 'sample.pdf' in the same directory
pdf_path = "sample.pdf"

# 2. Extract and Process Text
print("Extracting text from PDF...")
raw_text = get_pdf_text(pdf_path)
text_chunks = get_text_chunks(raw_text)

# 3. Create Vector Store
print("Creating Vector Store...")
vector_store = get_vector_store(text_chunks)
print("Vector Store Created Successfully!")

Extracting text from PDF...
Creating Vector Store...
Vector Store Created Successfully!


In [22]:
# 4. Ask a Question
user_question = "What is the main topic of the document?"

print(f"Question: {user_question}")
docs = vector_store.similarity_search(user_question, k=5)
chain = get_conversational_chain()

context = "\n".join([doc.page_content for doc in docs])
response = chain.invoke(
    {"context": context, "question": user_question}
)

print("\nReply:")
print(response)

Question: What is the main topic of the document?



Reply:
The main topic of the document is the application of AI and prompt engineering in digital learning environments.

The document details:
*   **Core Principles of Prompt Engineering and Structured Prompting**: Defining prompt engineering and outlining key techniques like Role Prompting, Few-Shot Prompting, Chain-of-Thought (CoT) Prompting, and Context Layering, specifically in educational setups.
*   **Current Applications of AI in Digital Learning Environments**: Discussing how AI is used in Intelligent Tutoring Systems (ITS), Automated Formative Assessment, and Operational Automation.
*   **Research Insights on Personalization and Structured Pathways**: Emphasizing effective learning through personalization, referencing Cognitive Load Theory (CLT) and Zone of Proximal Development (ZPD).
*   **AI in Opportunity Creation and Content Generation**: Highlighting AI's role in speeding up educational program development, analyzing job market data for course modules, and generating str